# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.8%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

## 1.準備

In [1]:
# omnicampus 実行用
!pip install ipywidgets


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cuda


# For Colab

In [ ]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

# For Local

In [2]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


In [3]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
#WORK_DIR = "/content/drive/MyDrive/weblab/DLBasics2025/Competition"
WORK_DIR = os.path.join(work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

c:\Users\dysk-\Desktop\Current task\EEG compe


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset





class ThingsEEGDataset(Dataset):
    # クラス共有変数としてEAの変換行列を保持する辞書を定義（Trainの統計量をVal/Testに引き継ぐため）
    _R_inv_sqrt_dict = None

    def __init__(self, split: str, use_vit: bool = True):
        assert split in ["train", "val", "test"]
        self.split = split
        self.use_vit = use_vit

        # データの読み込み
        self.X = np.load(f"data/{split}/eeg.npy").astype(np.float32)

        # trial-wise z-score
        self.X = (self.X - self.X.mean(axis=-1, keepdims=True)) / (
            self.X.std(axis=-1, keepdims=True) + 1e-6
        )
        self.X = np.clip(self.X, -5, 5)

        # 被験者インデックス (0~9)
        self.subject = (
            np.load(f"data/{split}/subject_idxs.npy").astype(np.int64) - 1
        )

        if split != "test":
            self.y = np.load(f"data/{split}/labels.npy").astype(np.int64)
        else:
            self.y = None

        if use_vit and split != "test":
            self.vit = np.load(f"data/{split}/vit_features.npy").astype(
                np.float32
            )
            self.vit = self.vit / (
                np.linalg.norm(self.vit, axis=1, keepdims=True) + 1e-6
            )
        else:
            self.vit = None

        # ==========================================
        # 🔥 Euclidean Alignment (EA) の計算と適用
        # ==========================================
        if split == "train":
            # Trainデータが初期化されるタイミングで、被験者ごとの共分散行列の逆数平方根を計算
            print("[EA Init] Trainデータから共分散行列の統計量を計算します...")
            ThingsEEGDataset._R_inv_sqrt_dict = {}
            unique_subjects = np.unique(self.subject)

            for sub in unique_subjects:
                idx = np.where(self.subject == sub)[0]
                X_sub = self.X[idx]  # shape: (N_sub, 17, 100)

                # 各試行の共分散行列 R = X @ X^T を計算して平均化
                cov_list = [np.dot(trial, trial.T) for trial in X_sub]
                R_sub = np.mean(cov_list, axis=0)

                # 数値安定化のための正則化
                R_sub += np.eye(R_sub.shape[0]) * 1e-6

                # 固有値分解で R^(-1/2) を算出
                eigvals, eigvecs = np.linalg.eigh(R_sub)
                eigvals = np.maximum(eigvals, 1e-10)
                R_inv_sqrt = np.dot(
                    eigvecs, np.dot(np.diag(1.0 / np.sqrt(eigvals)), eigvecs.T)
                )

                # 辞書に保存
                ThingsEEGDataset._R_inv_sqrt_dict[sub] = R_inv_sqrt.astype(
                    np.float32
                )
            print("✅ [EA Init] すべての被験者の R_inv_sqrt 計算が完了しました。")

        # 各試行データに対してその場でEA変換（空間白色化）を適用
        if ThingsEEGDataset._R_inv_sqrt_dict is not None:
            print(f"[{split}] EEGデータにEA変換を適用中...")
            for i in range(len(self.X)):
                sub_id = self.subject[i]
                if sub_id in ThingsEEGDataset._R_inv_sqrt_dict:
                    # R^(-1/2) @ X_i
                    self.X[i] = np.dot(
                        ThingsEEGDataset._R_inv_sqrt_dict[sub_id], self.X[i]
                    )
            print(f"✅ [{split}] EA変換の適用が完了しました。")
        else:
            print(
                f"⚠️ [{split}] Warning: Trainデータがまだ初期化されていないため、EAは適用されませんでした。"
            )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        subject = torch.tensor(self.subject[idx], dtype=torch.long)

        if self.split == "test":
            return x, subject

        y = torch.tensor(self.y[idx], dtype=torch.long)

        if self.use_vit:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            return x, subject, y, vit

        return x, subject, y
    

class ThingsEEGDatasetVITPCA(ThingsEEGDataset):
    def __init__(self, split: str, use_vit: bool = True, pca_dim: int = 256):
        super().__init__(split=split, use_vit=False)

        self.use_vit = use_vit
        self.pca_dim = pca_dim

        if use_vit and split != "test":
            self.vit = np.load(f"data/{split}/vit_pca{pca_dim}_features.npy").astype(np.float32)
            self.vit = self.vit / (
                np.linalg.norm(self.vit, axis=1, keepdims=True) + 1e-6
            )
            print(f"[{split}] vit_pca{pca_dim}:", self.vit.shape)
        else:
            self.vit = None

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        subject = torch.tensor(self.subject[idx], dtype=torch.long)

        if self.split == "test":
            return x, subject

        y = torch.tensor(self.y[idx], dtype=torch.long)

        if self.use_vit:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            return x, subject, y, vit

        return x, subject, y

class ThingsEEGDatasetVITCLIP(ThingsEEGDataset):
    def __init__(self, split: str, use_vit: bool = True, use_clip: bool = True):
        super().__init__(split=split, use_vit=use_vit)

        self.use_clip = use_clip

        if use_clip and split != "test":
            self.clip = np.load(f"data/{split}/clip_features.npy").astype(np.float32)
            self.clip = self.clip / (
                np.linalg.norm(self.clip, axis=1, keepdims=True) + 1e-6
            )
            print(f"[{split}] clip:", self.clip.shape)
        else:
            self.clip = None

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        subject = torch.tensor(self.subject[idx], dtype=torch.long)

        if self.split == "test":
            return x, subject

        y = torch.tensor(self.y[idx], dtype=torch.long)

        if self.use_vit and self.use_clip:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            clip = torch.tensor(self.clip[idx], dtype=torch.float32)
            return x, subject, y, vit, clip

        if self.use_vit:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            return x, subject, y, vit

        return x, subject, y

# 2.5 Load Config file

In [16]:
del run_dir

In [17]:
from pathlib import Path
from datetime import datetime
import json
import shutil

# ===== 読み込むconfigを指定 =====
#CONFIG_PATH =  Path("configs/baseline.json")
#CONFIG_PATH =  Path("configs/clip_m5_5.json")
#CONFIG_PATH =  Path("configs/baseline_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip_SubjectEmbedding.json")
#CONFIG_PATH =  Path("configs/b_baseline_eeg_to_vit_mse_cos.json")
CONFIG_PATH =  Path("configs/exp-eeg-to-vit-regression-ea.json") 


print(f"Loading config from: {CONFIG_PATH}")
#CONFIG_PATH = work_dir + CONFIG_PATH

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

# ===== configから変数に反映 =====
RUN_NAME = config["run_name"]
seed = config["seed"]
lr = config["lr"]
batch_size = config["batch_size"]
epochs = config["epochs"]
model_name = config["model_name"]
optimizer_name = config["optimizer"]
scheduler_name = config["scheduler"]

# ===== 保存先作成 =====
if "run_dir" not in globals():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    run_dir = Path("outputs") / f"{timestamp}_{RUN_NAME}"
    run_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(CONFIG_PATH, run_dir / "config.json")

print(f"Run directory: {run_dir}")

Loading config from: configs\exp-eeg-to-vit-regression-ea.json
Run directory: outputs\20260612_1726_exp-eeg-to-vit-regression-ea


# Load image_features data

In [18]:
from pathlib import Path
import numpy as np

feature_path = work_dir + "/data/features/vit_image_features.npy"
path_txt = work_dir + "/data/features/vit_image_paths.txt"
print(feature_path)


features = np.load(feature_path)

with open(path_txt) as f:
    feature_paths = [p.strip() for p in f.readlines()]

print(features.shape)
print(len(feature_paths))
print(feature_paths[0])

c:\Users\dysk-\Desktop\Current task\EEG compe/data/features/vit_image_features.npy
(5940, 768)
5940
00001_aardvark/aardvark_01b.jpg


In [7]:
# path -> feature の辞書
feature_dict = {
    p: feat
    for p, feat in zip(feature_paths, features)
}

def make_trial_image_features(split):
    path_file = work_dir + f"/data/{split}/image_paths.txt"

    with open(path_file) as f:
        trial_paths = [p.strip() for p in f.readlines()]

    trial_features = np.stack([
        feature_dict[p]
        for p in trial_paths
    ])

    return trial_features

train_img_feats = make_trial_image_features("train")
val_img_feats = make_trial_image_features("val")

print(train_img_feats.shape)
print(val_img_feats.shape)

(118800, 768)
(59400, 768)


In [8]:
np.save(work_dir + "/data/train/vit_features.npy", train_img_feats)
np.save(work_dir + "/data/val/vit_features.npy", val_img_feats)

## 3.ベースラインモデル

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class EEGToViTCLIPBaseline(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=768,
        clip_dim=512,
    ):
        super().__init__()

        self.encoder = EEGNetEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim
        print("hidden_dim:", hidden_dim)

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, vit_dim),
        )

        self.clip_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, clip_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        return F.normalize(z, dim=1)

    def forward_clip(self, x, subject):
        h = self.encode(x, subject)
        z = self.clip_head(h)
        return F.normalize(z, dim=1)

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits

class MidSelfAttentionBlock(nn.Module):
    """
    軽量な時間方向self-attention block。
    入力: (B, T, C)
    出力: (B, T, C)

    residual_scaleを小さく初期化して、
    初期状態ではB案EEGNetに近い挙動から始める。
    """
    def __init__(
        self,
        dim=128,
        num_heads=4,
        ff_dim=256,
        dropout=0.15,
        residual_scale_init=1e-3,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, dim),
        )
        self.drop2 = nn.Dropout(dropout)

        self.gamma_attn = nn.Parameter(torch.tensor(residual_scale_init))
        self.gamma_ffn = nn.Parameter(torch.tensor(residual_scale_init))

    def forward(self, x):
        # x: (B, T, C)
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(
            x_norm,
            x_norm,
            x_norm,
            need_weights=False,
        )
        x = x + self.gamma_attn * self.drop1(attn_out)

        ffn_out = self.ffn(self.norm2(x))
        x = x + self.gamma_ffn * self.drop2(ffn_out)

        return x


class EEGToViTPCABaseline(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=256,
    ):
        super().__init__()

        self.encoder = EEGNetEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim
        print("hidden_dim:", hidden_dim)

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, vit_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        return F.normalize(z, dim=1)

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits



class EEGNetMidSelfAttentionEncoder(nn.Module):
    def __init__(
        self,
        num_channels=17,
        num_times=100,
        dropout=0.25,
        attn_dropout=0.15,
    ):
        super().__init__()

        # ===== B案best相当 =====
        F1 = 64
        D = 2
        F2 = F1 * D  # 128

        self.F1 = F1
        self.D = D
        self.F2 = F2

        self.temporal = nn.Sequential(
            nn.Conv2d(
                1,
                F1,
                kernel_size=(1, 25),
                padding=(0, 12),
                bias=False,
            ),
            nn.BatchNorm2d(F1),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(num_channels, 1),
                groups=F1,
                bias=False,
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),  # T: 100 -> 25
            nn.Dropout(dropout),
        )

        self.mid_attn = MidSelfAttentionBlock(
            dim=F1 * D,
            num_heads=4,
            ff_dim=256,
            dropout=attn_dropout,
            residual_scale_init=1e-3,
        )

        self.separable = nn.Sequential(
            nn.Conv2d(
                F1 * D,
                F1 * D,
                kernel_size=(1, 15),
                padding=(0, 7),
                groups=F1 * D,
                bias=False,
            ),
            nn.Conv2d(
                F1 * D,
                F2,
                kernel_size=(1, 1),
                bias=False,
            ),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),  # T: 25 -> 6
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_times)
            out = self._forward_features_4d(dummy)
            self.out_dim = out.flatten(1).shape[1]

        print("mid self-attention encoder out_dim:", self.out_dim)

    def _forward_features_4d(self, x):
        # x: (B, 1, C, T)
        h = self.temporal(x)
        h = self.spatial(h)      # (B, 128, 1, 25)

        # 時間方向token列に変換
        h_tok = h.squeeze(2).transpose(1, 2)  # (B, 25, 128)
        h_tok = self.mid_attn(h_tok)          # (B, 25, 128)

        # Conv2d用に戻す
        h = h_tok.transpose(1, 2).unsqueeze(2)  # (B, 128, 1, 25)

        h = self.separable(h)  # (B, 128, 1, 6)
        return h

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        h = self._forward_features_4d(x)
        h = h.flatten(1)
        return h


class EEGToViTMidSelfAttention(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=768,
    ):
        super().__init__()

        self.encoder = EEGNetMidSelfAttentionEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim
        print("hidden_dim:", hidden_dim)

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, vit_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        z = F.normalize(z, dim=1)
        return z

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits

In [22]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size: int = 3,
        p_drop: float = 0.1,
    ) -> None:
        super().__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim

        self.conv0 = nn.Conv1d(in_dim, out_dim, kernel_size, padding="same")
        self.conv1 = nn.Conv1d(out_dim, out_dim, kernel_size, padding="same")
        # self.conv2 = nn.Conv1d(out_dim, out_dim, kernel_size) # , padding="same")

        self.batchnorm0 = nn.BatchNorm1d(num_features=out_dim)
        self.batchnorm1 = nn.BatchNorm1d(num_features=out_dim)

        self.dropout = nn.Dropout(p_drop)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if self.in_dim == self.out_dim:
            X = self.conv0(X) + X  # skip connection
        else:
            X = self.conv0(X)

        X = F.gelu(self.batchnorm0(X))

        X = self.conv1(X) + X  # skip connection
        X = F.gelu(self.batchnorm1(X))

        # X = self.conv2(X)
        # X = F.glu(X, dim=-2)

        return self.dropout(X)


class BasicConvClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        seq_len: int,
        in_channels: int,
        hid_dim: int = 128
    ) -> None:
        super().__init__()

        self.blocks = nn.Sequential(
            ConvBlock(in_channels, hid_dim),
            ConvBlock(hid_dim, hid_dim),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            Rearrange("b d 1 -> b d"),
            nn.Linear(hid_dim, num_classes),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """_summary_
        Args:
            X ( b, c, t ): _description_
        Returns:
            X ( b, num_classes ): _description_
        """
        X = self.blocks(X)

        return self.head(X)
    


class EEGNetClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_channels: int,
        seq_len: int,
        F1: int = 32,
        D: int = 2,
        F2: int = 64,
        dropout: float = 0.5,
        subject_emb_dim: int = 16,
        num_subjects: int = 10,
    ):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 15), padding=(0, 7), bias=False),
            nn.BatchNorm2d(F1),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 15), padding=(0, 7),
                      groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        with torch.no_grad():
            dummy = torch.zeros(1, num_channels, seq_len)
            feat = self._forward_features(dummy)
            feat_dim = feat.shape[1]

        self.classifier = nn.Linear(feat_dim + subject_emb_dim, num_classes)

    def _forward_features(self, x):
        x = x.unsqueeze(1)  # (batch, 1, channels, time)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return x

    def forward(self, x, subject_idxs):
        x = self._forward_features(x)
        subject_emb = self.subject_embedding(subject_idxs)
        x = torch.cat([x, subject_emb], dim=1)
        return self.classifier(x)


import torch
import torch.nn as nn
import torch.nn.functional as F


class EEGNetEncoder(nn.Module):
    def __init__(self, num_channels=17, num_times=100, dropout=0.25):
        super().__init__()

        # ===== B案 best相当 =====
        F1 = 64
        D = 2
        F2 = F1 * D  # 128

        self.net = nn.Sequential(
            nn.Conv2d(
                1,
                F1,
                kernel_size=(1, 25),
                padding=(0, 12),
                bias=False,
            ),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(num_channels, 1),
                groups=F1,
                bias=False,
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),

            nn.Conv2d(
                F1 * D,
                F1 * D,
                kernel_size=(1, 15),
                padding=(0, 7),
                groups=F1 * D,
                bias=False,
            ),
            nn.Conv2d(
                F1 * D,
                F2,
                kernel_size=(1, 1),
                bias=False,
            ),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_times)
            out = self.net(dummy)
            self.out_dim = out.flatten(1).shape[1]

        print("EEGNetEncoder out_dim:", self.out_dim)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        h = self.net(x)
        h = h.flatten(1)
        return h


class EEGToViTBaseline(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=768,
    ):
        super().__init__()

        self.encoder = EEGNetEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim
        print("hidden_dim:", hidden_dim)

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, vit_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        z = F.normalize(z, dim=1)
        return z

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits

# Set Seed

In [7]:
import random
import numpy as np
import torch

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(seed)

## 4.訓練実行

In [9]:
def mse_cos_loss(pred, target, alpha=0.5):
    pred = F.normalize(pred, dim=1)
    target = F.normalize(target, dim=1)

    mse = F.mse_loss(pred, target)
    cos_loss = 1.0 - F.cosine_similarity(pred, target, dim=1).mean()

    loss = alpha * mse + (1.0 - alpha) * cos_loss
    return loss, mse.detach(), cos_loss.detach()

In [25]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim

RUN_NAME = "j_f1_64_vit_pca256_regression"

train_ds = ThingsEEGDatasetVITPCA("train", use_vit=True, pca_dim=256)
val_ds = ThingsEEGDatasetVITPCA("val", use_vit=True, pca_dim=256)

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTPCABaseline(vit_dim=256).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
)

best_val_loss = float("inf")

for epoch in range(30):
    model.train()

    train_loss = 0.0
    train_vit_mse = 0.0
    train_vit_cos_loss = 0.0
    train_vit_cos = 0.0

    for x, subject, y, vit in tqdm(train_loader, desc=f"ViT-PCA pretrain {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        vit = vit.to(device)

        optimizer.zero_grad()

        pred_vit = model.forward_vit(x, subject)

        loss, mse, cos_loss = mse_cos_loss(
            pred_vit,
            vit,
            alpha=0.5,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        with torch.no_grad():
            vit_cos = F.cosine_similarity(
                F.normalize(pred_vit, dim=1),
                F.normalize(vit, dim=1),
                dim=1,
            ).mean()

        bs = x.size(0)
        train_loss += loss.item() * bs
        train_vit_mse += mse.item() * bs
        train_vit_cos_loss += cos_loss.item() * bs
        train_vit_cos += vit_cos.item() * bs

    scheduler.step()

    train_loss /= len(train_ds)
    train_vit_mse /= len(train_ds)
    train_vit_cos_loss /= len(train_ds)
    train_vit_cos /= len(train_ds)

    model.eval()

    val_loss = 0.0
    val_vit_mse = 0.0
    val_vit_cos_loss = 0.0
    val_vit_cos = 0.0

    with torch.no_grad():
        for x, subject, y, vit in val_loader:
            x = x.to(device)
            subject = subject.to(device)
            vit = vit.to(device)

            pred_vit = model.forward_vit(x, subject)

            loss, mse, cos_loss = mse_cos_loss(
                pred_vit,
                vit,
                alpha=0.5,
            )

            vit_cos = F.cosine_similarity(
                F.normalize(pred_vit, dim=1),
                F.normalize(vit, dim=1),
                dim=1,
            ).mean()

            bs = x.size(0)
            val_loss += loss.item() * bs
            val_vit_mse += mse.item() * bs
            val_vit_cos_loss += cos_loss.item() * bs
            val_vit_cos += vit_cos.item() * bs

    val_loss /= len(val_ds)
    val_vit_mse /= len(val_ds)
    val_vit_cos_loss /= len(val_ds)
    val_vit_cos /= len(val_ds)

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"train_mse={train_vit_mse:.5f} | "
        f"train_cos_loss={train_vit_cos_loss:.5f} | "
        f"train_vit_cos={train_vit_cos:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"val_mse={val_vit_mse:.5f} | "
        f"val_cos_loss={val_vit_cos_loss:.5f} | "
        f"val_vit_cos={val_vit_cos:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "model_j_vit_pca256_pretrained.pt")
        print(f"saved: model_j_vit_pca256_pretrained.pt | val_loss={best_val_loss:.5f}")

[EA Init] Trainデータから共分散行列の統計量を計算します...
✅ [EA Init] すべての被験者の R_inv_sqrt 計算が完了しました。
[train] EEGデータにEA変換を適用中...
✅ [train] EA変換の適用が完了しました。
[train] vit_pca256: (118800, 256)
[val] EEGデータにEA変換を適用中...
✅ [val] EA変換の適用が完了しました。
[val] vit_pca256: (59400, 256)
EEGNetEncoder out_dim: 768
hidden_dim: 784


ViT-PCA pretrain 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=0.47450 | train_mse=0.00736 | train_cos_loss=0.94165 | train_vit_cos=0.05835 | val_loss=0.46295 | val_mse=0.00718 | val_cos_loss=0.91873 | val_vit_cos=0.08127
saved: model_j_vit_pca256_pretrained.pt | val_loss=0.46295


ViT-PCA pretrain 2:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 02 | train_loss=0.46184 | train_mse=0.00716 | train_cos_loss=0.91652 | train_vit_cos=0.08348 | val_loss=0.45914 | val_mse=0.00712 | val_cos_loss=0.91117 | val_vit_cos=0.08883
saved: model_j_vit_pca256_pretrained.pt | val_loss=0.45914


...

epoch 28 | train_loss=0.42830 | train_mse=0.00664 | train_cos_loss=0.84996 | train_vit_cos=0.15004 | val_loss=0.44092 | val_mse=0.00684 | val_cos_loss=0.87501 | val_vit_cos=0.12499
saved: model_j_vit_pca256_pretrained.pt | val_loss=0.44092


ViT-PCA pretrain 29:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 29 | train_loss=0.42822 | train_mse=0.00664 | train_cos_loss=0.84980 | train_vit_cos=0.15020 | val_loss=0.44091 | val_mse=0.00684 | val_cos_loss=0.87499 | val_vit_cos=0.12501
saved: model_j_vit_pca256_pretrained.pt | val_loss=0.44091


ViT-PCA pretrain 30:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 30 | train_loss=0.42806 | train_mse=0.00664 | train_cos_loss=0.84949 | train_vit_cos=0.15051 | val_loss=0.44089 | val_mse=0.00684 | val_cos_loss=0.87494 | val_vit_cos=0.12506
saved: model_j_vit_pca256_pretrained.pt | val_loss=0.44089


In [12]:
train_ds_ft = ThingsEEGDatasetVITCLIP("train", use_vit=False, use_clip=False)
val_ds_ft = ThingsEEGDatasetVITCLIP("val", use_vit=False, use_clip=False)

train_loader_ft = DataLoader(
    train_ds_ft,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader_ft = DataLoader(
    val_ds_ft,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTCLIPBaseline().to(device)
model.load_state_dict(
    torch.load("model_i_vit_clip_pretrained.pt", map_location=device)
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 3e-4},
        {"params": model.subject_emb.parameters(), "lr": 3e-4},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

best_val_acc = 0.0

for epoch in range(50):
    model.train()

    train_loss = 0.0
    train_correct = 0

    for x, subject, y in tqdm(train_loader_ft, desc=f"ViT+CLIP finetune {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model.forward_cls(x, subject)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = x.size(0)
        train_loss += loss.item() * bs
        train_correct += (logits.argmax(dim=1) == y).sum().item()

    scheduler.step()

    train_loss /= len(train_ds_ft)
    train_acc = train_correct / len(train_ds_ft)

    model.eval()

    val_loss = 0.0
    val_correct = 0

    with torch.no_grad():
        for x, subject, y in val_loader_ft:
            x = x.to(device)
            subject = subject.to(device)
            y = y.to(device)

            logits = model.forward_cls(x, subject)
            loss = criterion(logits, y)

            bs = x.size(0)
            val_loss += loss.item() * bs
            val_correct += (logits.argmax(dim=1) == y).sum().item()

    val_loss /= len(val_ds_ft)
    val_acc = val_correct / len(val_ds_ft)

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | train_acc={train_acc:.5f} | "
        f"val_loss={val_loss:.5f} | val_acc={val_acc:.5f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "model_i_vit_clip_finetuned_best.pt")
        torch.save(model.state_dict(), "model_best.pt")
        print(f"saved: model_i_vit_clip_finetuned_best.pt | val_acc={best_val_acc:.5f}")

[EA Init] Trainデータから共分散行列の統計量を計算します...
✅ [EA Init] すべての被験者の R_inv_sqrt 計算が完了しました。
[train] EEGデータにEA変換を適用中...
✅ [train] EA変換の適用が完了しました。
[val] EEGデータにEA変換を適用中...
✅ [val] EA変換の適用が完了しました。
EEGNetEncoder out_dim: 768
hidden_dim: 784


C:\Users\dysk-\AppData\Local\Temp\ipykernel_12532\2188433343.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("model_i_vit_clip_pretrained.pt", map_location=d

ViT+CLIP finetune 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=1.36543 | train_acc=0.46813 | val_loss=1.31451 | val_acc=0.49715
saved: model_i_vit_clip_finetuned_best.pt | val_acc=0.49715


ViT+CLIP finetune 2:   0%|          | 0/465 [00:00<?, ?it/s]

...

epoch 48 | train_loss=1.18375 | train_acc=0.55962 | val_loss=1.25902 | val_acc=0.52296


ViT+CLIP finetune 49:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 49 | train_loss=1.18428 | train_acc=0.56052 | val_loss=1.25925 | val_acc=0.52352


ViT+CLIP finetune 50:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 50 | train_loss=1.18187 | train_acc=0.55974 | val_loss=1.25973 | val_acc=0.52293


## 5.評価

In [13]:
test_ds = ThingsEEGDatasetVITCLIP("test", use_vit=False, use_clip=False)

test_loader = DataLoader(
    test_ds,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTCLIPBaseline().to(device)
model.load_state_dict(
    torch.load("model_i_vit_clip_finetuned_best.pt", map_location=device)
)

model.eval()

all_probs = []

with torch.no_grad():
    for x, subject in tqdm(test_loader, desc="predict ViT+CLIP"):
        x = x.to(device)
        subject = subject.to(device)

        logits = model.forward_cls(x, subject)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
y_pred = all_probs.argmax(axis=1)

np.save("submission.npy", all_probs)
np.save("probs_i_f1_64_vit_clip_multitask_w05.npy", all_probs)
np.save("y_pred_i_f1_64_vit_clip_multitask_w05.npy", y_pred)

print("submission:", all_probs.shape)
print("ndim:", all_probs.ndim)
print("row sum:", all_probs.sum(axis=1)[:5])
print("pred counts:", np.bincount(y_pred, minlength=5))
print("first 50 pred:", y_pred[:50])

[test] EEGデータにEA変換を適用中...
✅ [test] EA変換の適用が完了しました。
EEGNetEncoder out_dim: 768
hidden_dim: 784


C:\Users\dysk-\AppData\Local\Temp\ipykernel_12532\958185324.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("model_i_vit_clip_finetuned_best.pt", map_locatio

predict ViT+CLIP:   0%|          | 0/117 [00:00<?, ?it/s]

submission: (59400, 5)
ndim: 2
row sum: [1.         0.99999994 0.99999994 1.         1.0000001 ]
pred counts: [12494 34624  6299  5073   910]
first 50 pred: [3 1 1 0 1 0 0 0 1 3 2 1 3 2 0 1 1 1 3 3 1 2 0 1 1 1 1 1 3 1 0 2 3 0 1 1 1
 1 0 0 1 1 1 1 2 1 0 1 1 1]


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [14]:
from zipfile import ZipFile
from datetime import datetime
from pathlib import Path

#timestamp = datetime.now().strftime("%Y%m%d_%H%M")
#run_dir = Path("outputs") / "20260611_0353_b_baseline_eeg_to_vit_mse_cos"
zip_name = run_dir / "submission.zip"



submission_path = run_dir / "submission.npy"
model_path = run_dir / "model_best.pt"
notebook_path = Path(work_dir) / "notebooks" / "DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile(zip_name, "w") as zf:
    zf.write(submission_path, arcname="submission.npy")
    zf.write(model_path, arcname="model_best.pt")
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb")

print(f"Created: {zip_name}")

with ZipFile(zip_name, "r") as zf:
    print(zf.namelist())

Created: outputs\20260612_1700_exp-eeg-to-vit-regression-ea\submission.zip
['submission.npy', 'model_best.pt', 'DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb']
